In [1]:
#Python Modules
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch 
from torch.utils.data import DataLoader

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

#ML Modules
import train.train_code as tr_c
import preprocess.data_frame_preprocess as prep
import models.model_bge_m3 as bge_m3
import models.model_xgboost as xgb_m3
import evaluate
import dataset.Train as multi_task

#Config FILE
import config_test_t2 as cfg

/home/andreoseo/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Transformer

## Check Save Paths

In [2]:
os.makedirs(cfg.paths.SAVE_DIR,exist_ok=True)

In [3]:
raw_df = pd.read_csv(cfg.paths.CSV_PATH)
df = prep.prepare_dataframe(raw_df,cfg) #New columns: lyrics_clean & gloss_clean

strat_key = (df['label_t1'].astype(str) + '_' +
             df['label_t3'].astype(str) + '_' +
             df[cfg.columns.LABEL_COLS_T2].astype(str).agg('-'.join, axis=1))

[FLAG]: Using Glosses for Training


In [4]:
df

,lyrics,resultado,lyrics_clean,gloss_clean,label_t1,type_hate,type_violence,type_sexualization,nothing,label_t3
0,La vida es un puta mierda\nLa vida es un puta ...,La letra está cargada de lenguaje extremadamen...,la vida es un puta mierda la vida es un puta m...,la letra está cargada de lenguaje extremadamen...,0,0,0,0,1,0
1,"No empuje señora\nBajo en la próxima, que ni u...",La letra critica fuertemente la hipocresía soc...,"no empuje señora bajo en la próxima, que ni us...",la letra critica fuertemente la hipocresía soc...,0,0,0,0,1,0
2,"No a mí, no así, co\nEs Lírico, escrito bajo l...",La letra es muy arrogante y contiene referenci...,"no a mí, no así, co es lírico, escrito bajo la...",la letra es muy arrogante y contiene referenci...,0,0,0,0,1,0
3,Si vuelvo a- Si vuelvo a reconciliarme con el ...,La letra describe la vida nocturna y el desgas...,si vuelvo a- si vuelvo a reconciliarme con el ...,la letra describe la vida nocturna y el desgas...,0,0,0,0,1,0
4,"Nunca se vive suficiente, le dijo ella al depe...",La canción es una profunda meditación sobre la...,"nunca se vive suficiente, le dijo ella al depe...",la canción es una profunda meditación sobre la...,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...
571,"(Yeah, yeah)\n(AMEX got the micro)\nEstá aleja...",El contenido es extremadamente sexual y vulgar...,"(yeah, yeah) (amex got the micro) está alejand...",el contenido es extremadamente sexual y vulgar...,0,0,0,0,1,0
572,"Lía, se acabó el amor, se murió esa flor\nNo s...",La letra promueve la venganza emocional y util...,"lía, se acabó el amor, se murió esa flor no se...",la letra promueve la venganza emocional y util...,0,0,0,0,1,0
573,Ah-ah-ah\nAh-ah-ah\nAhhh\nTe digo la verdad (U...,La canción expresa un deseo intenso y urgente ...,ah-ah-ah ah-ah-ah ahhh te digo la verdad (uh) ...,la canción expresa un deseo intenso y urgente ...,0,0,0,0,1,0
574,"Casi las 8, salto a la ducha\nNo, no me puedo ...",La letra se enfoca en la sexualización del ent...,"casi las 8, salto a la ducha no, no me puedo r...",la letra se enfoca en la sexualización del ent...,0,0,0,0,1,0


## Load Models

In [5]:
#Load Transformer Model
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer,model = bge_m3.load_tokenizer_model(cfg,device)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 95798.40it/s]


## [DEFINE] Dataset and Loader

In [6]:
test_ds = multi_task.MultiTaskDataset(df, cfg, tokenizer, cfg.tokenization.MAX_LEN,
                            head_len=cfg.tokenization.HEAD_LEN,
                            model_prefix=cfg.paths.MODEL_PREFIX)

test_loader = DataLoader(test_ds, batch_size=cfg.train.BATCH_SIZE,
                         num_workers=2, pin_memory=True)

# [LOAD] Weights (BGE M3 & XGBOOST)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cargar pesos entrenados
#tokenizer, model = load_tokenizer_model(cfg, device)
model.load_state_dict(
    torch.load(
        os.path.join(
            '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_v3_only_t2',
            'model_weights.pt'
        ),
        map_location=device
    )
)

<All keys matched successfully>

In [8]:
X_test, y_t1_test, y_t2_test, y_t3_test =     xgb_m3.extract_embeddings_and_labels(model, test_loader, device)
#X_test_emb  = extract_embeddings(model, test_loader,  device)

In [37]:
import joblib
import numpy as np 

xgb_t1 = joblib.load('/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t1.pkl')
xgb_t3 = joblib.load('/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t3.pkl')


# cargar t2 como dict de clasificadores individuales
clfs_t2 = {
    col: joblib.load(f'/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t2_{col}.pkl')
    for col in cfg.columns.LABEL_COLS_T2
}


# inferencia
pred_t1 = xgb_t1.predict(X_test)
pred_t3 = xgb_t3.predict(X_test)
pred_t2 = np.column_stack([
    clfs_t2[col].predict(X_test)
    for col in cfg.columns.LABEL_COLS_T2
])


print(f"""
    pred1_shape -> {pred_t1.shape}
    pred2_shape -> {pred_t2.shape}
    pred3_shape -> {pred_t3.shape}
    """)



    pred1_shape -> (576,)
    pred2_shape -> (576, 4)
    pred3_shape -> (576,)
    


array([[0, 0, 1, 0],
       [0, 0, 0, 1],
       [1, 0, 0, 0],
       ...,
       [0, 0, 0, 1],
       [1, 0, 1, 0],
       [1, 1, 1, 0]], shape=(576, 4))

In [38]:
pred_t1 #1 -> M & 0 -> NM
pred_t2 #[0,0,0,0] -> acorde cfg -> ['type_sexualization', 'type_violence', 'type_hate', 'nothing']
pred_t3 #1 -> Y & 0 -> N

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,

In [39]:
# mapeos inversos
map_t1 = {1: 'M', 0: 'NM'}
map_t3 = {1: 'Y', 0: 'N'}

results_df = df.copy()
results_df['pred_t1'] = [map_t1[p] for p in pred_t1]
results_df['pred_t3'] = [map_t3[p] for p in pred_t3]
for i, col in enumerate(cfg.columns.LABEL_COLS_T2):
    results_df['pred_t2_' + col] = pred_t2[:, i].astype(int)

In [40]:
results_df.to_csv("results_xgboost.csv")

In [35]:
results_df

,lyrics,resultado,lyrics_clean,gloss_clean,label_t1,type_hate,type_violence,type_sexualization,nothing,label_t3,pred_t1,pred_t3,pred_t2_type_sexualization,pred_t2_type_violence,pred_t2_type_hate,pred_t2_nothing
0,La vida es un puta mierda\nLa vida es un puta ...,La letra está cargada de lenguaje extremadamen...,la vida es un puta mierda la vida es un puta m...,la letra está cargada de lenguaje extremadamen...,0,0,0,0,1,0,M,N,0,0,1,0
1,"No empuje señora\nBajo en la próxima, que ni u...",La letra critica fuertemente la hipocresía soc...,"no empuje señora bajo en la próxima, que ni us...",la letra critica fuertemente la hipocresía soc...,0,0,0,0,1,0,NM,N,0,0,0,1
2,"No a mí, no así, co\nEs Lírico, escrito bajo l...",La letra es muy arrogante y contiene referenci...,"no a mí, no así, co es lírico, escrito bajo la...",la letra es muy arrogante y contiene referenci...,0,0,0,0,1,0,M,N,1,0,0,0
3,Si vuelvo a- Si vuelvo a reconciliarme con el ...,La letra describe la vida nocturna y el desgas...,si vuelvo a- si vuelvo a reconciliarme con el ...,la letra describe la vida nocturna y el desgas...,0,0,0,0,1,0,M,N,1,0,0,0
4,"Nunca se vive suficiente, le dijo ella al depe...",La canción es una profunda meditación sobre la...,"nunca se vive suficiente, le dijo ella al depe...",la canción es una profunda meditación sobre la...,0,0,0,0,1,0,NM,N,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,"(Yeah, yeah)\n(AMEX got the micro)\nEstá aleja...",El contenido es extremadamente sexual y vulgar...,"(yeah, yeah) (amex got the micro) está alejand...",el contenido es extremadamente sexual y vulgar...,0,0,0,0,1,0,M,N,1,1,1,0
572,"Lía, se acabó el amor, se murió esa flor\nNo s...",La letra promueve la venganza emocional y util...,"lía, se acabó el amor, se murió esa flor no se...",la letra promueve la venganza emocional y util...,0,0,0,0,1,0,M,Y,1,1,1,0
573,Ah-ah-ah\nAh-ah-ah\nAhhh\nTe digo la verdad (U...,La canción expresa un deseo intenso y urgente ...,ah-ah-ah ah-ah-ah ahhh te digo la verdad (uh) ...,la canción expresa un deseo intenso y urgente ...,0,0,0,0,1,0,NM,N,0,0,0,1
574,"Casi las 8, salto a la ducha\nNo, no me puedo ...",La letra se enfoca en la sexualización del ent...,"casi las 8, salto a la ducha no, no me puedo r...",la letra se enfoca en la sexualización del ent...,0,0,0,0,1,0,M,Y,1,0,1,0
